Won't be deployed until I can move away from Render (also why its a jupyter notebook)

Check if valid:
    - Is screenshot of TEC
    - Is on results screen

Check for:
    - Both sides: APM, DPM, SR, Score
    - One side: Time

In [1]:
# imports

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

2025-11-14 10:42:08.897709: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 10:42:09.011494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-14 10:42:10.554696: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# constants
INP_DIR:str = "data/validation"
DATA_DIR:str = "data/validation_processed/training"
TEST_DIR:str = "data/validation_processed/testing"
PROC_DIR:str = "data/validation_processed/unsorted"
SEED:int = 136
TRAIN_TEST_SPLIT:float = 0.2


In [3]:
# create directories

if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR)

if not os.path.exists(f"{TEST_DIR}/0_invalid"):
    os.makedirs(f"{TEST_DIR}/0_invalid")

if not os.path.exists(f"{TEST_DIR}/1_tec_results"):
    os.makedirs(f"{TEST_DIR}/1_tec_results")


if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

if not os.path.exists(f"{DATA_DIR}/0_invalid"):
    os.makedirs(f"{DATA_DIR}/0_invalid")

if not os.path.exists(f"{DATA_DIR}/1_tec_results"):
    os.makedirs(f"{DATA_DIR}/1_tec_results")

if not os.path.exists(PROC_DIR):
    os.makedirs(PROC_DIR)

if not os.path.exists(f"{PROC_DIR}/0_invalid"):
    os.makedirs(f"{PROC_DIR}/0_invalid")

if not os.path.exists(f"{PROC_DIR}/1_tec_results"):
    os.makedirs(f"{PROC_DIR}/1_tec_results")

In [4]:
# convert all data to b/w

def bw_image(path_in:str, class_dir:str, name:str) -> None:
    img = Image.open(path_in)

    bw_img = img.convert("1")

    bw_img.save(f"{PROC_DIR}/{class_dir}/bw_{name}")

for class_dir in os.listdir(INP_DIR):

    path:str = f"{INP_DIR}/{class_dir}"

    for file in os.listdir(path):
        bw_image(f"{path}/{file}", class_dir, file)

In [5]:
# now we need to split the data
files = []
labels = []


# first, index everything 
for class_dir in os.listdir(PROC_DIR):
    class_path = f"{PROC_DIR}/{class_dir}"

    # makes a list of all the files in the path
    img_files = glob.glob(f"{class_path}/*")

    label:int = 1 if class_dir.startswith("1") else 0

    files.extend(img_files)
    labels.extend([label]*len(img_files))

# split them with train test split

train_val_files, test_files, y, y = train_test_split(
    files,
    labels,
    test_size=TRAIN_TEST_SPLIT,
    random_state=SEED,
    stratify=labels
)

for file_path in train_val_files:
    new_path = file_path.replace("/unsorted/", "/training/")
    shutil.copy(file_path, new_path)

for file_path in test_files:
    new_path = file_path.replace("/unsorted/", "/testing/")
    shutil.copy(file_path, new_path)

In [6]:
# load the data MobileNetV3Small

IMG_SIZE:tuple[int, int] = (224, 224)
BATCH_SIZE:int = 32

print("Loading and resizing to", IMG_SIZE)

train_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


valid_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_tec_results.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_tec_results = train_tec_results.cache().prefetch(buffer_size=AUTOTUNE)
valid_tec_results = valid_tec_results.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n Found classes: {class_names}")

Loading and resizing to (224, 224)
Found 127 files belonging to 2 classes.
Using 102 files for training.


I0000 00:00:1763138564.131750   77740 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 127 files belonging to 2 classes.
Using 25 files for validation.

 Found classes: ['0_invalid', '1_tec_results']


In [ ]:
# Construct a Mobile Net 
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape= IMG_SIZE + (1,),
    alpha=1.0, # determines how much to cut down the model, 1 is fully intact
    minimalistic=False,
    include_top=False, # removes classification layer so we can add our own
    weights="imagenet",
    input_tensor=None,
        classes=1000,
    pooling=None,
    dropout_rate=0.2,
    classifier_activation="softmax",
    include_preprocessing=True,
)

for layer in base_model.layers:
    layer.trainable = False

# binary classification outputs
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediciton_layer = tf.keras.layers.Dense(1, activation="sigmoid")

inputs = tf.keras.Input(shape=IMG_SIZE + (1,))

base_model_out = base_model(inputs, training=False)
pooled_out = global_average_layer(base_model_out)
dropped_out = tf.keras.layers.Dropout(0.2)(pooled_out)

outputs = prediciton_layer(dropped_out)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           577 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 939,697 (3.58 MB)

 Trainable params: 577 (2.25 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [26]:
print("It's training time!")

checkpointer = tf.keras.callbacks.ModelCheckpoint(
  filepath='tec_check_weights.keras',
  monitor='accuracy',
  verbose=1,
  save_best_only=True
)


It's training time!


In [27]:
model.fit(
    train_tec_results,
    epochs=500,
    validation_data=valid_tec_results,
    callbacks=checkpointer,
    verbose=1
)

Epoch 1/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 990ms/step - accuracy: 0.5448 - loss: 0.7286
Epoch 1: accuracy improved from None to 0.54902, saving model to tec_check_weights.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.5490 - loss: 0.7160 - val_accuracy: 0.5600 - val_loss: 0.6717
Epoch 2/500
2/4 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.5703 - loss: 0.6948
Epoch 2: accuracy improved from 0.54902 to 0.56863, saving model to tec_check_weights.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 158ms/step - accuracy: 0.5686 - loss: 0.6910 - val_accuracy: 0.5600 - val_loss: 0.6690
Epoch 3/500
1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5312 - loss: 0.7415
Epoch 3: accuracy did not improve from 0.56863
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5588 - loss: 0.6912 - val_accuracy: 0.5600 - val_loss: 0.6664
Epoch 4/500
1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4375 - loss: 0.7815
Epoch 4: accuracy did not improve from 0.56863
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accur

In [29]:
# load testing

test_set = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_paths = test_set.file_paths # this is to see which it misclassified

test_set = test_set.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

loss, acc = model.evaluate(test_set, verbose=2)

print(f"\nFinal Test Accuracy: {acc * 100:.2f}%")

print(test_paths)

Found 32 files belonging to 2 classes.
1/1 - 0s - 249ms/step - accuracy: 0.9062 - loss: 0.3432

Final Test Accuracy: 90.62%
['data/validation_processed/testing/0_invalid/bw_520414-Spitzer_Space_Telescope-space-galaxy-NASA.jpg', 'data/validation_processed/testing/0_invalid/bw_Code_cLVDLRK5u7.png', 'data/validation_processed/testing/0_invalid/bw_Photos_CarjngwKGP.png', 'data/validation_processed/testing/0_invalid/bw_Photos_UIsEqz2BgM.png', 'data/validation_processed/testing/0_invalid/bw_Photos_jJdJ7asYzs.png', 'data/validation_processed/testing/0_invalid/bw_Photos_lFXRlPEaKG.png', 'data/validation_processed/testing/0_invalid/bw_Photos_we0V8pSquG.png', 'data/validation_processed/testing/0_invalid/bw_msedge_KDxbW3s7oe.png', 'data/validation_processed/testing/0_invalid/bw_msedge_OKeuhBhYSl.png', 'data/validation_processed/testing/0_invalid/bw_msedge_W2SUcsqofo.png', 'data/validation_processed/testing/0_invalid/bw_msedge_hMEZOdBfim.png', 'data/validation_processed/testing/0_invalid/bw_msedge

In [30]:
# see which ones it failed

predictions = model.predict(test_set)

predicted_labels = np.round(predictions).flatten().astype(int)

true_labels = np.concatenate([y.numpy() for x, y in test_set], axis=0).flatten().astype(int)

misclassified_indices = np.where(predicted_labels != true_labels)[0]

for i in misclassified_indices:
    print(f"\n\npredicted as {predicted_labels[i]} | actually is {true_labels[i]}")
    print(test_paths[i])


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


predicted as 1 | actually is 0
data/validation_processed/testing/0_invalid/bw_steamwebhelper_mkZoZMIPdH.png


predicted as 0 | actually is 1
data/validation_processed/testing/1_tec_results/bw_Discord_9l84CUi6WD.png


predicted as 0 | actually is 1
data/validation_processed/testing/1_tec_results/bw_Photos_ZK8pIaOWl3.png


2025-11-14 10:55:47.321689: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
